# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geerro-aleks/My-FlyRank-AI-Internship-ML-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This task is scoring because this model calculates an actionable "Opportunity Score" to measure lost potential traffic for each page. By assigning this relative numeric value, content teams can easily rank pages to focus on the highest-leverage rewrite opportunities.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The model predicts Recoverable Clicks (or CTR Gap), which quantifies lost traffic by scaling a page's CTR deficit against its position tier's benchmark by its total impressions. This target is created using a defined rule applied to observed outcome data: it combines actual observed metrics—$Impressions$, $Clicks$, and $Position$—with a computed site-wide benchmark rule to measure a page's traffic potential.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

The success metric uses Spearman’s Rank Correlation Coefficient ($\rho \ge 0.75$) because human content workflows require accurate relative prioritization rather than exact numerical precision. Search Console data contains inherent noise, such as varying SERP features and competitor movement, making a perfect correlation ($\rho \ge 0.90$) unfeasible and prone to overfitting. A threshold of 0.75 proves strong monotonic alignment, guaranteeing that the model reliably ranks the highest-leverage rewrite opportunities at the top.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# 1. LOAD THE STARTER DATASET
# ------------------------------------------------------------------------------
# Replace 'path_to_data.csv' with your local CSV file path or Hugging Face dataset path
# e.g., url = "https://huggingface.co/datasets/flyrank/flyrank-data/raw/main/search_console_data.csv"
try:
    df = pd.read_csv("work/data/flyrank_starter_data.csv")
except FileNotFoundError:
    # Creating a synthetic starter dataframe matching FlyRank structure for demonstration
    np.random.seed(42)
    data = {
        'url': [f'https://example.com/page-{i}' for i in range(1, 101)],
        'position': np.random.uniform(1.0, 20.0, 100),
        'impressions': np.random.randint(500, 50000, 100),
        'clicks': np.random.randint(10, 2000, 100),
    }
    df = pd.DataFrame(data)

# 2. FEATURE ENGINEERING & BENCHMARK CALCULATION
# ------------------------------------------------------------------------------
# Calculate observed CTR for each page
df['observed_ctr'] = df['clicks'] / df['impressions']

# Group position into discrete tiers (e.g., Tier 1: Pos 1-3, Tier 2: Pos 4-10, etc.)
def assign_position_tier(pos):
    if pos <= 3.0:
        return 'Tier 1 (Top 3)'
    elif pos <= 10.0:
        return 'Tier 2 (Striking Distance: 4-10)'
    elif pos <= 20.0:
        return 'Tier 3 (Page 2: 11-20)'
    else:
        return 'Tier 4 (Page 3+)'

df['position_tier'] = df['position'].apply(assign_position_tier)

# Compute mean CTR benchmark per position tier across the entire dataset
tier_benchmarks = df.groupby('position_tier')['observed_ctr'].transform('mean')
df['benchmark_ctr'] = tier_benchmarks

# 3. CONSTRUCT THE TARGET VARIABLE (OPPORTUNITY SCORE)
# ------------------------------------------------------------------------------
# Formula: Recoverable Clicks = (Benchmark CTR - Observed CTR) * Impressions
df['ctr_gap'] = df['benchmark_ctr'] - df['observed_ctr']

# Only calculate positive recoverable clicks (if page underperforms tier average)
df['recoverable_clicks'] = np.maximum(0, df['ctr_gap'] * df['impressions'])

# Opportunity Score normalized between 0 and 100 for actionable ranking
max_recovery = df['recoverable_clicks'].max()
df['opportunity_score'] = (df['recoverable_clicks'] / max_recovery) * 100

# 4. DISPLAY UNIT OF ANALYSIS DATAFRAME
# ------------------------------------------------------------------------------
print("=== UNIT OF ANALYSIS DATAFRAME (1 Row = 1 Page) ===")
print(f"Total Pages Analyzed: {len(df)}\n")

# Display top 5 highest priority rewrite opportunities
display_cols = [
    'url', 'position', 'position_tier', 'impressions',
    'observed_ctr', 'benchmark_ctr', 'recoverable_clicks', 'opportunity_score'
]

top_opportunities = df.sort_values(by='opportunity_score', ascending=False)[display_cols].head(5)
top_opportunities


=== UNIT OF ANALYSIS DATAFRAME (1 Row = 1 Page) ===
Total Pages Analyzed: 100



,url,position,position_tier,impressions,observed_ctr,benchmark_ctr,recoverable_clicks,opportunity_score
72,https://example.com/page-73,1.104920,Tier 1 (Top 3),47217,0.019548,0.196344,8347.786841,100.000000
6,https://example.com/page-7,2.103589,Tier 1 (Top 3),40842,0.003820,0.196344,7863.092194,94.193735
29,https://example.com/page-30,1.882558,Tier 1 (Top 3),26605,0.005751,0.196344,5070.738990,60.743513
98,https://example.com/page-99,1.482963,Tier 1 (Top 3),26870,0.017454,0.196344,4806.770219,57.581372
67,https://example.com/page-68,16.241743,Tier 3 (Page 2: 11-20),47145,0.011009,0.100564,4222.083993,50.577286


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Static if/else rules fail because search performance depends on complex, non-linear interactions between rank position, SERP features, search intent, and shifting user behavior that simple logic cannot capture. An ML model dynamically adapts to these multi-variable patterns and shifting site baselines without requiring constant manual rule recalibration.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.